# FMA Data Preprocessing

This notebook cleans the FMA metadata, emits per-subset CSVs, and turns the `fma_small` MP3s into log-mel spectrograms saved to disk.

Two cleaned subsets are written:
- **small** — `subset == "small"` (≈8 000 tracks). Used for spectrograms and any small-only model. Drops missing/undecodable audio + the `FAILED` track ids from `creation.ipynb`.
- **medium** — `subset ∈ {small, medium}` (≈25 000 tracks; FMA medium is a superset of small). RF-friendly: drops null genre, the `FAILED` list, and the few corrupt small MP3s we already know about. No per-track audio decode check (we don't have `fma_medium/` on disk).

Spectrograms are only extracted for the small subset. Random Forest can train on either subset by picking the right CSV.

## 1. Setup

Run once if any of these are missing:

```python
%pip install torch torchaudio pandas numpy tqdm soundfile
```

To specify dataset directory - set environment variable:
```python
os.environ["DATASET_DIR"] = "/datasets/kaspoas"

```


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import os

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

# Resolve paths from either the project root or the code/ subdirectory.
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]

if "DATASET_DIR" in os.environ:
    PROJECT_CANDIDATES.append(Path(os.environ["DATASET_DIR"]))
    
for candidate in PROJECT_CANDIDATES:
    audio_dir = candidate / "fma_medium"
    metadata_path = candidate / "fma_metadata" / "tracks.csv"
    if audio_dir.exists() and metadata_path.exists():
        PROJECT_DIR = candidate.resolve()
        AUDIO_DIR = audio_dir.resolve()
        METADATA_PATH = metadata_path.resolve()
        break
else:
    raise FileNotFoundError("Could not find fma_medium/ and fma_metadata/tracks.csv.")

# Where the cleaned CSVs and spectrograms get written.
PROCESSED_DIR = PROJECT_DIR / "fma_preprocessed_medium"
SPECTROGRAM_DIR = PROCESSED_DIR / "spectrograms"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPECTROGRAM_DIR.mkdir(parents=True, exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print({
    "PROJECT_DIR": str(PROJECT_DIR),
    "AUDIO_DIR": str(AUDIO_DIR),
    "METADATA_PATH": str(METADATA_PATH),
    "PROCESSED_DIR": str(PROCESSED_DIR),
    "SPECTROGRAM_DIR": str(SPECTROGRAM_DIR),
    "device": device,
})

{'PROJECT_DIR': '/home-mscluster/skapenga/fma_project/fma', 'AUDIO_DIR': '/datasets/skapenga/kaspoas/fma_medium', 'METADATA_PATH': '/home-mscluster/skapenga/fma_project/fma/data/fma_metadata/tracks.csv', 'PROCESSED_DIR': '/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium', 'SPECTROGRAM_DIR': '/home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms', 'device': 'cpu'}


## 2. Load `tracks.csv`

`tracks.csv` uses a two-row header, so columns come back as a `MultiIndex` like `("track", "genre_top")`. We pull only the fields we need and flatten them into a normal DataFrame.

In [2]:
def track_id_to_audio_path(track_id: int) -> Path:
    track_id = int(track_id)
    filename = f"{track_id:06d}.mp3"
    return AUDIO_DIR / filename[:3] / filename


tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

# tracks.index is named "track_id" (from index_col=0). Drop it before building metadata_all
# so the resulting frame has track_id only as a column, not also as an index name —
# otherwise sort_values("track_id") later raises "both an index level and a column label".
tracks_flat = tracks.reset_index(drop=True)

metadata_all = pd.DataFrame({
    "track_id": tracks.index.astype(int).to_numpy(),
    "subset": tracks_flat[("set", "subset")].to_numpy(),
    "split": tracks_flat[("set", "split")].to_numpy(),
    "genre": tracks_flat[("track", "genre_top")].to_numpy(),
    "duration": tracks_flat[("track", "duration")].to_numpy(),
    "bit_rate": tracks_flat[("track", "bit_rate")].to_numpy(),
    "title": tracks_flat[("track", "title")].to_numpy(),
    "artist": tracks_flat[("artist", "name")].to_numpy(),
})
metadata_all["audio_path"] = metadata_all["track_id"].apply(track_id_to_audio_path)

print(f"Total tracks in tracks.csv: {len(metadata_all):,}")
print(metadata_all["subset"].value_counts())
metadata_all.head()

Total tracks in tracks.csv: 106,574
subset
large     81574
medium    17000
small      8000
Name: count, dtype: int64


,track_id,subset,split,genre,duration,bit_rate,title,artist,audio_path
0,2,small,training,Hip-Hop,168,256000,Food,AWOL,/datasets/skapenga/kaspoas/fma_medium/000/0000...
1,3,medium,training,Hip-Hop,237,256000,Electric Ave,AWOL,/datasets/skapenga/kaspoas/fma_medium/000/0000...
2,5,small,training,Hip-Hop,206,256000,This World,AWOL,/datasets/skapenga/kaspoas/fma_medium/000/0000...
3,10,small,training,Pop,161,192000,Freeway,Kurt Vile,/datasets/skapenga/kaspoas/fma_medium/000/0000...
4,20,large,training,NaN,311,256000,Spiritual Level,Nicky Cook,/datasets/skapenga/kaspoas/fma_medium/000/0000...


## 3. Clean each subset

Each subset drops the same three kinds of entries:

- **null** — tracks with no `genre_top`.
- **empty** — tracks whose MP3 is missing on disk or zero bytes.
- **failed** — track ids `creation.ipynb` already flagged as broken (ffmpeg/header issues, librosa "filter pass-band lies beyond Nyquist", etc.).

We use `creation.ipynb`'s `keep(...)` helper so it is obvious what each filter drops. The medium frame skips the empty-file check for medium-only tracks because we don't have `fma_medium/` locally — they stay in the CSV so the random forest can still train on the tabular features.

In [3]:
# Same `keep` helper used in creation.ipynb.
def keep(mask, df, label):
    old = len(df)
    df = df[mask]
    print(f"{label:42s} {old - len(df):>5d} dropped, {len(df):>5d} left")
    return df


def is_file_nonempty(path: Path) -> bool:
    try:
        return path.exists() and path.stat().st_size > 0
    except OSError:
        return False


# Track ids that creation.ipynb documents as feature-extraction failures
# (ffmpeg header issues, librosa "filter pass-band lies beyond Nyquist", etc.).
FAILED = [
    1440, 26436, 28106, 29166, 29167, 29168, 29169, 29170, 29171, 29172,
    29173, 29179, 38903, 43903, 56757, 57603, 59361, 62095, 62954, 62956,
    62957, 62959, 62971, 75461, 80015, 86079, 92345, 92346, 92347, 92348,
    92349, 92350, 92351, 92352, 92353, 92354, 92355, 92356, 92357, 92358,
    92359, 92360, 92361, 96426, 104623, 106719, 109714, 114448, 114501, 114528,
    115235, 117759, 118003, 118004, 127827, 130296, 130298, 131076, 135804, 136486,
    144769, 144770, 144771, 144773, 144774, 144775, 144776, 144777, 144778, 152204,
    154923,
]


print("=== Cleaning small ===")
metadata = metadata_all.copy()

metadata = keep(metadata["subset"] == "small", metadata, "subset == small")
metadata = keep(metadata["genre"].notna(), metadata, "genre not null")
metadata = keep(metadata["audio_path"].map(is_file_nonempty), metadata, "audio file present and non-empty")
metadata = keep(~metadata["track_id"].isin(FAILED), metadata, "not in FAILED list")

metadata = metadata.sort_values("track_id").reset_index(drop=True)
genres_small_sorted = sorted(metadata["genre"].unique())
genre_to_idx_small = {g: i for i, g in enumerate(genres_small_sorted)}
metadata["label"] = metadata["genre"].map(genre_to_idx_small).astype(int)

print("\nSmall genre distribution:")
print(metadata["genre"].value_counts().sort_index())
print("\nSmall split distribution:")
print(metadata["split"].value_counts())

=== Cleaning small ===
subset == small                            98574 dropped,  8000 left
genre not null                                 0 dropped,  8000 left


audio file present and non-empty               0 dropped,  8000 left
not in FAILED list                             0 dropped,  8000 left

Small genre distribution:
genre
Electronic       1000
Experimental     1000
Folk             1000
Hip-Hop          1000
Instrumental     1000
International    1000
Pop              1000
Rock             1000
Name: count, dtype: int64

Small split distribution:
split
training      6400
validation     800
test           800
Name: count, dtype: int64


In [4]:
print("=== Cleaning medium ===")
metadata_medium = metadata_all.copy()

# FMA convention: medium ⊃ small. Use isin so the cleaned medium frame has all 25k tracks
# (8000 small + 17000 medium-only), matching the FMA paper's medium definition.
metadata_medium = keep(
    metadata_medium["subset"].isin(["small", "medium"]),
    metadata_medium,
    "subset in {small, medium}",
)
metadata_medium = keep(metadata_medium["genre"].notna(), metadata_medium, "genre not null")
metadata_medium = keep(~metadata_medium["track_id"].isin(FAILED), metadata_medium, "not in FAILED list")

# We don't have fma_medium/ on disk, so the empty-file check only runs for the small portion;
# medium-only tracks stay in the CSV regardless (tabular features.csv covers them anyway).
# is_small = metadata_medium["subset"] == "small"
# file_ok_or_medium_only = metadata_medium["audio_path"].map(is_file_nonempty) | (~is_small)
# metadata_medium = keep(file_ok_or_medium_only, metadata_medium, "small-portion audio present and non-empty")

# Now you do want to extract medium audio spectrograms
metadata_medium = keep(
    metadata_medium["audio_path"].map(is_file_nonempty),
    metadata_medium,
    "audio file present and non-empty",
)

metadata_medium = metadata_medium.sort_values("track_id").reset_index(drop=True)
genres_medium_sorted = sorted(metadata_medium["genre"].unique())
genre_to_idx_medium = {g: i for i, g in enumerate(genres_medium_sorted)}
metadata_medium["label"] = metadata_medium["genre"].map(genre_to_idx_medium).astype(int)

print("\nMedium genre distribution:")
print(metadata_medium["genre"].value_counts().sort_index())
print("\nMedium split distribution:")
print(metadata_medium["split"].value_counts())

=== Cleaning medium ===
subset in {small, medium}                  81574 dropped, 25000 left
genre not null                                 0 dropped, 25000 left
not in FAILED list                             0 dropped, 25000 left


audio file present and non-empty               0 dropped, 25000 left

Medium genre distribution:
genre
Blues                    74
Classical               619
Country                 178
Easy Listening           21
Electronic             6314
Experimental           2251
Folk                   1519
Hip-Hop                2201
Instrumental           1350
International          1018
Jazz                    384
Old-Time / Historic     510
Pop                    1186
Rock                   7103
Soul-RnB                154
Spoken                  118
Name: count, dtype: int64

Medium split distribution:
split
training      19922
test           2573
validation     2505
Name: count, dtype: int64


## 4. Save the cleaned CSVs

For each subset we write:
- `tracks_clean_{subset}.csv` — the full cleaned frame.
- `tracks_clean_{subset}_{split}.csv` — per-split slices so training notebooks can read just what they need.
- `genre_to_idx_{subset}.csv` — the genre → label mapping.

In [5]:
def save_clean_csvs(frame: pd.DataFrame, subset_name: str, genre_to_idx: dict[str, int]) -> None:
    base_path = PROCESSED_DIR / f"tracks_clean_{subset_name}.csv"
    frame.to_csv(base_path, index=False)
    print(f"Wrote {base_path} ({len(frame):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = frame[frame["split"] == split_name]
        split_path = PROCESSED_DIR / f"tracks_clean_{subset_name}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    genre_map_path = PROCESSED_DIR / f"genre_to_idx_{subset_name}.csv"
    pd.DataFrame(
        sorted(genre_to_idx.items(), key=lambda kv: kv[1]),
        columns=["genre", "label"],
    ).to_csv(genre_map_path, index=False)
    print(f"Wrote {genre_map_path}")


save_clean_csvs(metadata, "small", genre_to_idx_small)
print()
save_clean_csvs(metadata_medium, "medium", genre_to_idx_medium)

Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_small.csv (8,000 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_small_training.csv (6,400 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_small_validation.csv (800 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_small_test.csv (800 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/genre_to_idx_small.csv



Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_medium.csv (25,000 rows)


Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_medium_training.csv (19,922 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_medium_validation.csv (2,505 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/tracks_clean_medium_test.csv (2,573 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/genre_to_idx_medium.csv


## 5. Spectrogram settings

Same numbers as `audio_encoder.ipynb` / `cnn_vs_transformer.ipynb` so the saved spectrograms are drop-in compatible:

- 16 kHz mono
- 10 s clip (centered) → 160 000 samples
- STFT: `n_fft=400`, `hop=160` (≈ 25 ms window, 10 ms hop)
- 64 mel bins, log power scale, per-clip mean/std normalization
- Output shape: `(1, 64, 1001)` saved as `float32`

In [6]:
SAMPLE_RATE = 16_000
CLIP_SECONDS = 10
NUM_SAMPLES = SAMPLE_RATE * CLIP_SECONDS
N_FFT = 400
HOP_LENGTH = 160
N_MELS = 64

mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
).to(device)
to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(device)


def load_clip(path: Path) -> torch.Tensor:
    """Read an MP3, convert to mono 16 kHz, center-crop or pad to NUM_SAMPLES."""
    audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)

    if sample_rate != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)

    if waveform.shape[1] > NUM_SAMPLES:
        start = (waveform.shape[1] - NUM_SAMPLES) // 2
        waveform = waveform[:, start:start + NUM_SAMPLES]
    elif waveform.shape[1] < NUM_SAMPLES:
        waveform = torch.nn.functional.pad(waveform, (0, NUM_SAMPLES - waveform.shape[1]))

    waveform = torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0)
    return waveform.clamp(-1.0, 1.0)


@torch.no_grad()
def waveform_to_spec(waveform: torch.Tensor) -> torch.Tensor:
    waveform = waveform.to(device)
    spec = to_db(mel_transform(waveform))
    spec = torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)
    mean = spec.mean(dim=(-2, -1), keepdim=True)
    std = spec.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
    spec = (spec - mean) / std
    return torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0).cpu()


# Sanity check on one clip.
sample_row = metadata_medium.iloc[0]
sample_wave = load_clip(sample_row["audio_path"])
sample_spec = waveform_to_spec(sample_wave)
print(f"waveform shape: {tuple(sample_wave.shape)}")
print(f"spectrogram shape: {tuple(sample_spec.shape)}")
print(f"sample track: {sample_row['track_id']} ({sample_row['genre']})")

waveform shape: (1, 160000)
spectrogram shape: (1, 64, 1001)
sample track: 2 (Hip-Hop)


## 6. Extract and save every spectrogram

Output layout mirrors `fma_small/`:

```
fma_preprocessed/
  spectrograms/
    000/000002.npy
    000/000005.npy
    ...
```

Already-extracted spectrograms are skipped so the cell is safe to re-run. Tracks that throw during decode/STFT are recorded in a final `dropped` list and removed from the manifest.

In [7]:
def track_id_to_spec_path(track_id: int) -> Path:
    filename = f"{int(track_id):06d}.npy"
    return SPECTROGRAM_DIR / filename[:3] / filename


# Toggle to False to recompute everything from scratch.
SKIP_EXISTING = True

spec_paths = []
dropped_rows = []
n_skipped = 0
n_written = 0

for row in tqdm(metadata_medium.itertuples(index=False), total=len(metadata_medium), desc="Extracting"):
    out_path = track_id_to_spec_path(row.track_id)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if SKIP_EXISTING and out_path.exists():
        spec_paths.append(str(out_path))
        n_skipped += 1
        continue

    try:
        waveform = load_clip(Path(row.audio_path))
        spec = waveform_to_spec(waveform).numpy().astype(np.float32)
    except Exception as exc:
        dropped_rows.append({
            "track_id": row.track_id,
            "audio_path": str(row.audio_path),
            "error": f"{type(exc).__name__}: {exc}",
        })
        spec_paths.append(None)
        continue

    np.save(out_path, spec)
    spec_paths.append(str(out_path))
    n_written += 1

print(f"\nWrote {n_written:,} new spectrograms, reused {n_skipped:,} existing, "
      f"dropped {len(dropped_rows):,} on error.")
if dropped_rows:
    print(pd.DataFrame(dropped_rows).head(10).to_string(index=False))

Extracting:   0%|          | 0/25000 [00:00<?, ?it/s]

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


[src/libmpg123/layer3.c:INT123_do_layer3():1844] error: dequantization failed!


[src/libmpg123/layer3.c:INT123_do_layer3():1774] error: part2_3_length (3264) too large for available bit count (3224)


...

[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...


[src/libmpg123/parse.c:do_readahead():1083] warning: Cannot read next header, a one-frame stream? Duh...



Wrote 24,980 new spectrograms, reused 0 existing, dropped 20 on error.
 track_id                                           audio_path                                                                                                                                                   error
     1486 /datasets/skapenga/kaspoas/fma_medium/001/001486.mp3 LibsndfileError: Error opening '/datasets/skapenga/kaspoas/fma_medium/001/001486.mp3': File does not exist or is not a regular file (possibly a pipe?).
     5574 /datasets/skapenga/kaspoas/fma_medium/005/005574.mp3 LibsndfileError: Error opening '/datasets/skapenga/kaspoas/fma_medium/005/005574.mp3': File does not exist or is not a regular file (possibly a pipe?).
    65753 /datasets/skapenga/kaspoas/fma_medium/065/065753.mp3 LibsndfileError: Error opening '/datasets/skapenga/kaspoas/fma_medium/065/065753.mp3': File does not exist or is not a regular file (possibly a pipe?).
    80391 /datasets/skapenga/kaspoas/fma_medium/080/080391.mp3 Libsn

## 7. Save the spectrogram manifest

`spectrograms_manifest.csv` is the canonical pointer a training notebook should consume: one row per usable clip with the `.npy` path, label, split, and original audio path.

In [8]:
manifest = metadata_medium.copy()
manifest["spectrogram_path"] = spec_paths
manifest = manifest[manifest["spectrogram_path"].notna()].reset_index(drop=True)

manifest_columns = [
    "track_id", "split", "genre", "label",
    "spectrogram_path", "audio_path",
    "duration", "bit_rate", "title", "artist",
]
manifest = manifest[manifest_columns]

manifest_path = PROCESSED_DIR / "spectrograms_manifest.csv"
manifest.to_csv(manifest_path, index=False)
print(f"Wrote {manifest_path} ({len(manifest):,} rows)")

for split_name in ("training", "validation", "test"):
    split_manifest = manifest[manifest["split"] == split_name]
    split_path = PROCESSED_DIR / f"spectrograms_manifest_{split_name}.csv"
    split_manifest.to_csv(split_path, index=False)
    print(f"Wrote {split_path} ({len(split_manifest):,} rows)")

if dropped_rows:
    dropped_path = PROCESSED_DIR / "spectrogram_extraction_errors.csv"
    pd.DataFrame(dropped_rows).to_csv(dropped_path, index=False)
    print(f"Wrote {dropped_path} ({len(dropped_rows):,} rows)")

Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms_manifest.csv (24,980 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms_manifest_training.csv (19,904 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms_manifest_validation.csv (2,504 rows)


Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrograms_manifest_test.csv (2,572 rows)
Wrote /home-mscluster/skapenga/fma_project/fma/fma_preprocessed_medium/spectrogram_extraction_errors.csv (20 rows)


## 8. Quick sanity check

Load one `.npy` back and confirm the shape/stats match the in-memory tensor we generated above.

In [9]:
first = manifest.iloc[0]
loaded = np.load(first["spectrogram_path"])
print(f"track_id: {first['track_id']} | genre: {first['genre']} | split: {first['split']}")
print(f"shape: {loaded.shape} | dtype: {loaded.dtype}")
print(f"mean: {loaded.mean():.4f} | std: {loaded.std():.4f} | "
      f"min: {loaded.min():.2f} | max: {loaded.max():.2f}")

track_id: 2 | genre: Hip-Hop | split: training
shape: (1, 64, 1001) | dtype: float32
mean: 0.0000 | std: 1.0000 | min: -6.00 | max: 3.76
